# RecoMart Data Validation & Profiling

## Overview
This notebook implements comprehensive data quality validation and profiling for the RecoMart recommendation pipeline. It identifies data quality issues, validates business rules, and generates detailed quality reports.

## Validation Scope
1. **Schema Validation**: Verify expected columns and data types
2. **Completeness**: Identify missing values in critical fields
3. **Uniqueness**: Detect duplicate records
4. **Range Checks**: Validate numeric values are within expected ranges
5. **Format Validation**: Check data formats (dates, IDs, categories)
6. **Referential Integrity**: Validate foreign key relationships

## Data Sources Validated
* User Interactions (`recomart.raw.user_interactions`)
* Transactions (`recomart.raw.transactions`)
* Products (`recomart.raw.products`)

## Quality Metrics
* Completeness score (% non-null values)
* Uniqueness score (% non-duplicate records)
* Validity score (% values within expected ranges)
* Overall quality score

## Outputs
* Detailed data quality report (PDF)
* Quality metrics summary
* Flagged records requiring remediation

In [0]:
# Import required libraries
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import logging
import os
import json
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Configure logging
log_dir = "/Workspace/Users/2025ae05415@wilp.bits-pilani.ac.in/RecoMart_Recommendation_Pipeline/logs"
os.makedirs(log_dir, exist_ok=True)
log_file = f"{log_dir}/validation.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('RecoMartValidation')

# Define project paths
PROJECT_ROOT = "/Workspace/Users/2025ae05415@wilp.bits-pilani.ac.in/RecoMart_Recommendation_Pipeline"
REPORT_DIR = f"{PROJECT_ROOT}/documentation"
os.makedirs(REPORT_DIR, exist_ok=True)

# Unity Catalog configuration
CATALOG_NAME = 'recomart'
SCHEMA_NAME = 'raw'

logger.info("="*80)
logger.info("RecoMart Data Validation Pipeline - Session Started")
logger.info(f"Timestamp: {datetime.now().isoformat()}")
logger.info("="*80)

print("✅ Setup complete - Validation framework initialized")
print(f"📁 Log file: {log_file}")
print(f"📊 Reports will be saved to: {REPORT_DIR}")

In [0]:
# Define validation rules for each data source

VALIDATION_RULES = {
    'user_interactions': {
        'table_name': f'{CATALOG_NAME}.{SCHEMA_NAME}.user_interactions',
        'required_columns': ['user_id', 'item_id', 'interaction_type', 'timestamp', 'session_id', 'device_type'],
        'not_null_columns': ['user_id', 'item_id', 'timestamp'],
        'categorical_columns': {
            'interaction_type': ['click', 'view', 'add_to_cart', 'purchase'],
            'device_type': ['mobile', 'web', 'tablet']
        },
        'date_columns': ['timestamp'],
        'unique_columns': ['user_id', 'item_id', 'timestamp', 'session_id'],  # Composite uniqueness
        'id_pattern': {
            'user_id': r'^U\d{4}$',
            'item_id': r'^P\d{4}$'
        }
    },
    'transactions': {
        'table_name': f'{CATALOG_NAME}.{SCHEMA_NAME}.transactions',
        'required_columns': ['transaction_id', 'user_id', 'item_id', 'quantity', 'price', 'rating', 'timestamp', 'payment_method'],
        'not_null_columns': ['transaction_id', 'user_id', 'item_id', 'price'],
        'unique_columns': ['transaction_id'],
        'range_checks': {
            'rating': (1, 5),
            'price': (0.01, 10000),
            'quantity': (1, 100)
        },
        'categorical_columns': {
            'payment_method': ['credit_card', 'debit_card', 'paypal', 'upi', 'wallet']
        },
        'date_columns': ['timestamp'],
        'id_pattern': {
            'transaction_id': r'^T\d{6}$',
            'user_id': r'^U\d{4}$',
            'item_id': r'^P\d{4}$'
        }
    },
    'products': {
        'table_name': f'{CATALOG_NAME}.{SCHEMA_NAME}.products',
        'required_columns': ['item_id', 'product_name', 'category', 'sub_category', 'brand', 'price', 'stock_status'],
        'not_null_columns': ['item_id', 'product_name', 'category', 'price'],
        'unique_columns': ['item_id'],
        'range_checks': {
            'price': (0.01, 10000),
            'popularity_score': (0, 100),
            'sentiment_score': (0, 1)
        },
        'categorical_columns': {
            'stock_status': ['in_stock', 'out_of_stock', 'limited_stock']
        },
        'id_pattern': {
            'item_id': r'^P\d{4}$'
        }
    }
}

print("✅ Validation rules defined for all data sources")
print(f"\n📋 Data sources to validate: {len(VALIDATION_RULES)}")
for source in VALIDATION_RULES.keys():
    print(f"  • {source}")

In [0]:
def validate_schema(df, source_name, rules):
    """
    Validate DataFrame schema against expected columns.
    """
    issues = []
    required_cols = rules.get('required_columns', [])
    actual_cols = df.columns
    
    missing_cols = set(required_cols) - set(actual_cols)
    if missing_cols:
        issues.append({
            'check': 'Schema Validation',
            'severity': 'CRITICAL',
            'issue': f"Missing required columns: {', '.join(missing_cols)}",
            'count': len(missing_cols)
        })
        logger.error(f"{source_name}: Missing columns {missing_cols}")
    
    return issues

def validate_completeness(df, source_name, rules):
    """
    Check for missing values in critical columns.
    """
    issues = []
    not_null_cols = rules.get('not_null_columns', [])
    
    for col in not_null_cols:
        if col in df.columns:
            null_count = df.filter(F.col(col).isNull()).count()
            total_count = df.count()
            
            if null_count > 0:
                null_pct = (null_count / total_count) * 100
                issues.append({
                    'check': 'Completeness',
                    'severity': 'HIGH' if null_pct > 5 else 'MEDIUM',
                    'issue': f"Column '{col}' has {null_count} null values ({null_pct:.2f}%)",
                    'count': null_count,
                    'column': col
                })
                logger.warning(f"{source_name}: {col} has {null_count} nulls")
    
    return issues

def validate_uniqueness(df, source_name, rules):
    """
    Check for duplicate records based on unique columns.
    """
    issues = []
    unique_cols = rules.get('unique_columns', [])
    
    if unique_cols:
        total_count = df.count()
        distinct_count = df.select(unique_cols).distinct().count()
        duplicate_count = total_count - distinct_count
        
        if duplicate_count > 0:
            dup_pct = (duplicate_count / total_count) * 100
            issues.append({
                'check': 'Uniqueness',
                'severity': 'HIGH',
                'issue': f"Found {duplicate_count} duplicate records ({dup_pct:.2f}%) based on {unique_cols}",
                'count': duplicate_count
            })
            logger.warning(f"{source_name}: {duplicate_count} duplicates found")
    
    return issues

def validate_ranges(df, source_name, rules):
    """
    Validate numeric columns are within expected ranges.
    """
    issues = []
    range_checks = rules.get('range_checks', {})
    
    for col, (min_val, max_val) in range_checks.items():
        if col in df.columns:
            out_of_range = df.filter(
                (F.col(col).isNotNull()) & 
                ((F.col(col) < min_val) | (F.col(col) > max_val))
            ).count()
            
            if out_of_range > 0:
                total = df.filter(F.col(col).isNotNull()).count()
                out_pct = (out_of_range / total) * 100 if total > 0 else 0
                issues.append({
                    'check': 'Range Validation',
                    'severity': 'HIGH',
                    'issue': f"Column '{col}' has {out_of_range} values outside range [{min_val}, {max_val}] ({out_pct:.2f}%)",
                    'count': out_of_range,
                    'column': col
                })
                logger.warning(f"{source_name}: {col} has {out_of_range} out-of-range values")
    
    return issues

def validate_categories(df, source_name, rules):
    """
    Validate categorical columns have only expected values.
    Fixed for Databricks Serverless - no RDD operations.
    """
    issues = []
    categorical_cols = rules.get('categorical_columns', {})
    
    for col, expected_values in categorical_cols.items():
        if col in df.columns:
            # Get distinct values using DataFrame operations (no RDD)
            distinct_df = df.select(col).filter(F.col(col).isNotNull()).distinct()
            actual_values = [row[0] for row in distinct_df.collect()]
            invalid_values = set(actual_values) - set(expected_values)
            
            if invalid_values:
                # Count records with invalid values
                invalid_count = df.filter(F.col(col).isin(list(invalid_values))).count()
                total = df.filter(F.col(col).isNotNull()).count()
                invalid_pct = (invalid_count / total) * 100 if total > 0 else 0
                
                issues.append({
                    'check': 'Categorical Validation',
                    'severity': 'MEDIUM',
                    'issue': f"Column '{col}' has invalid values: {', '.join(map(str, invalid_values))} ({invalid_pct:.2f}%)",
                    'count': invalid_count,
                    'column': col
                })
                logger.warning(f"{source_name}: {col} has invalid values {invalid_values}")
    
    return issues

def calculate_quality_score(issues, total_records):
    """
    Calculate overall quality score based on issues found.
    """
    if not issues:
        return 100.0
    
    # Weight issues by severity
    severity_weights = {'CRITICAL': 10, 'HIGH': 5, 'MEDIUM': 2, 'LOW': 1}
    
    total_penalty = 0
    for issue in issues:
        severity = issue.get('severity', 'LOW')
        count = issue.get('count', 1)
        weight = severity_weights.get(severity, 1)
        
        # Calculate penalty as percentage of affected records
        penalty = (count / total_records) * weight if total_records > 0 else weight
        total_penalty += penalty
    
    # Cap penalty at 100
    total_penalty = min(total_penalty, 100)
    
    quality_score = max(0, 100 - total_penalty)
    return round(quality_score, 2)

print("✅ Validation functions defined (Fixed for Serverless):")
print("  • validate_schema: Schema and column checks")
print("  • validate_completeness: Missing value detection")
print("  • validate_uniqueness: Duplicate identification")
print("  • validate_ranges: Numeric range validation")
print("  • validate_categories: Categorical value validation (no RDD)")
print("  • calculate_quality_score: Quality scoring")

In [0]:
print("\n" + "="*80)
print("🔍 VALIDATING USER INTERACTIONS")
print("="*80)

validation_results = {}

try:
    # Load data
    table_name = VALIDATION_RULES['user_interactions']['table_name']
    df_interactions = spark.table(table_name)
    total_records = df_interactions.count()
    
    print(f"\n📊 Total records: {total_records:,}")
    logger.info(f"Validating {total_records} user interaction records")
    
    # Run validations
    all_issues = []
    rules = VALIDATION_RULES['user_interactions']
    
    print("\n🔍 Running validation checks...")
    
    # 1. Schema validation
    print("  • Schema validation...")
    all_issues.extend(validate_schema(df_interactions, 'user_interactions', rules))
    
    # 2. Completeness check
    print("  • Completeness check...")
    all_issues.extend(validate_completeness(df_interactions, 'user_interactions', rules))
    
    # 3. Uniqueness check
    print("  • Uniqueness check...")
    all_issues.extend(validate_uniqueness(df_interactions, 'user_interactions', rules))
    
    # 4. Categorical validation
    print("  • Categorical validation...")
    all_issues.extend(validate_categories(df_interactions, 'user_interactions', rules))
    
    # Calculate quality score
    quality_score = calculate_quality_score(all_issues, total_records)
    
    validation_results['user_interactions'] = {
        'total_records': total_records,
        'issues': all_issues,
        'quality_score': quality_score,
        'status': 'PASS' if quality_score >= 80 else 'FAIL'
    }
    
    # Display results
    print(f"\n📊 Validation Results:")
    print(f"  Issues found: {len(all_issues)}")
    print(f"  Quality score: {quality_score}/100")
    print(f"  Status: {validation_results['user_interactions']['status']}")
    
    if all_issues:
        print(f"\n⚠️  Issues Details:")
        for i, issue in enumerate(all_issues, 1):
            print(f"  {i}. [{issue['severity']}] {issue['issue']}")
    
except Exception as e:
    logger.error(f"Error validating user interactions: {str(e)}")
    print(f"\n❌ Validation failed: {str(e)}")
    validation_results['user_interactions'] = {
        'status': 'ERROR',
        'error': str(e)
    }

In [0]:
print("\n" + "="*80)
print("🔍 VALIDATING TRANSACTIONS")
print("="*80)

try:
    # Load data
    table_name = VALIDATION_RULES['transactions']['table_name']
    df_transactions = spark.table(table_name)
    total_records = df_transactions.count()
    
    print(f"\n📊 Total records: {total_records:,}")
    logger.info(f"Validating {total_records} transaction records")
    
    # Run validations
    all_issues = []
    rules = VALIDATION_RULES['transactions']
    
    print("\n🔍 Running validation checks...")
    
    # 1. Schema validation
    print("  • Schema validation...")
    all_issues.extend(validate_schema(df_transactions, 'transactions', rules))
    
    # 2. Completeness check
    print("  • Completeness check...")
    all_issues.extend(validate_completeness(df_transactions, 'transactions', rules))
    
    # 3. Uniqueness check
    print("  • Uniqueness check...")
    all_issues.extend(validate_uniqueness(df_transactions, 'transactions', rules))
    
    # 4. Range validation
    print("  • Range validation...")
    all_issues.extend(validate_ranges(df_transactions, 'transactions', rules))
    
    # 5. Categorical validation
    print("  • Categorical validation...")
    all_issues.extend(validate_categories(df_transactions, 'transactions', rules))
    
    # Calculate quality score
    quality_score = calculate_quality_score(all_issues, total_records)
    
    validation_results['transactions'] = {
        'total_records': total_records,
        'issues': all_issues,
        'quality_score': quality_score,
        'status': 'PASS' if quality_score >= 80 else 'FAIL'
    }
    
    # Display results
    print(f"\n📊 Validation Results:")
    print(f"  Issues found: {len(all_issues)}")
    print(f"  Quality score: {quality_score}/100")
    print(f"  Status: {validation_results['transactions']['status']}")
    
    if all_issues:
        print(f"\n⚠️  Issues Details:")
        for i, issue in enumerate(all_issues, 1):
            print(f"  {i}. [{issue['severity']}] {issue['issue']}")
    
except Exception as e:
    logger.error(f"Error validating transactions: {str(e)}")
    print(f"\n❌ Validation failed: {str(e)}")
    validation_results['transactions'] = {
        'status': 'ERROR',
        'error': str(e)
    }

In [0]:
print("\n" + "="*80)
print("🔍 VALIDATING PRODUCTS")
print("="*80)

try:
    # Load data
    table_name = VALIDATION_RULES['products']['table_name']
    df_products = spark.table(table_name)
    total_records = df_products.count()
    
    print(f"\n📊 Total records: {total_records:,}")
    logger.info(f"Validating {total_records} product records")
    
    # Run validations
    all_issues = []
    rules = VALIDATION_RULES['products']
    
    print("\n🔍 Running validation checks...")
    
    # 1. Schema validation
    print("  • Schema validation...")
    all_issues.extend(validate_schema(df_products, 'products', rules))
    
    # 2. Completeness check
    print("  • Completeness check...")
    all_issues.extend(validate_completeness(df_products, 'products', rules))
    
    # 3. Uniqueness check
    print("  • Uniqueness check...")
    all_issues.extend(validate_uniqueness(df_products, 'products', rules))
    
    # 4. Range validation
    print("  • Range validation...")
    all_issues.extend(validate_ranges(df_products, 'products', rules))
    
    # 5. Categorical validation
    print("  • Categorical validation...")
    all_issues.extend(validate_categories(df_products, 'products', rules))
    
    # Calculate quality score
    quality_score = calculate_quality_score(all_issues, total_records)
    
    validation_results['products'] = {
        'total_records': total_records,
        'issues': all_issues,
        'quality_score': quality_score,
        'status': 'PASS' if quality_score >= 80 else 'FAIL'
    }
    
    # Display results
    print(f"\n📊 Validation Results:")
    print(f"  Issues found: {len(all_issues)}")
    print(f"  Quality score: {quality_score}/100")
    print(f"  Status: {validation_results['products']['status']}")
    
    if all_issues:
        print(f"\n⚠️  Issues Details:")
        for i, issue in enumerate(all_issues, 1):
            print(f"  {i}. [{issue['severity']}] {issue['issue']}")
    
except Exception as e:
    logger.error(f"Error validating products: {str(e)}")
    print(f"\n❌ Validation failed: {str(e)}")
    validation_results['products'] = {
        'status': 'ERROR',
        'error': str(e)
    }

In [0]:
print("\n" + "="*80)
print("📋 DATA QUALITY SUMMARY REPORT")
print("="*80)
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n")

# Overall statistics
total_sources = len(validation_results)
passed_sources = sum(1 for v in validation_results.values() if v.get('status') == 'PASS')
failed_sources = sum(1 for v in validation_results.values() if v.get('status') == 'FAIL')
error_sources = sum(1 for v in validation_results.values() if v.get('status') == 'ERROR')

total_records = sum(v.get('total_records', 0) for v in validation_results.values())
total_issues = sum(len(v.get('issues', [])) for v in validation_results.values())

avg_quality_score = np.mean([v.get('quality_score', 0) for v in validation_results.values() if 'quality_score' in v])

print("📊 Overall Statistics:")
print(f"  • Total data sources: {total_sources}")
print(f"  • Passed validation: {passed_sources}")
print(f"  • Failed validation: {failed_sources}")
print(f"  • Errors encountered: {error_sources}")
print(f"  • Total records validated: {total_records:,}")
print(f"  • Total issues found: {total_issues}")
print(f"  • Average quality score: {avg_quality_score:.2f}/100")
print("\n")

# Detailed results by source
print("📋 Detailed Results by Data Source:")
print("-" * 80)

for source_name, result in validation_results.items():
    status = result.get('status', 'UNKNOWN')
    quality_score = result.get('quality_score', 0)
    total_recs = result.get('total_records', 0)
    issues = result.get('issues', [])
    
    status_icon = "✅" if status == 'PASS' else "❌" if status == 'FAIL' else "⚠️"
    
    print(f"\n{status_icon} {source_name.upper().replace('_', ' ')}")
    print(f"   Status: {status}")
    print(f"   Records: {total_recs:,}")
    print(f"   Quality Score: {quality_score}/100")
    print(f"   Issues Found: {len(issues)}")
    
    if issues:
        # Group issues by severity
        by_severity = defaultdict(list)
        for issue in issues:
            by_severity[issue.get('severity', 'UNKNOWN')].append(issue)
        
        for severity in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']:
            if severity in by_severity:
                print(f"\n   {severity} Issues ({len(by_severity[severity])}):")
                for issue in by_severity[severity][:5]:  # Show top 5
                    print(f"     • {issue['issue']}")
                if len(by_severity[severity]) > 5:
                    print(f"     ... and {len(by_severity[severity]) - 5} more")

print("\n" + "="*80)

# Save report to file
report_data = {
    'report_date': datetime.now().isoformat(),
    'overall_stats': {
        'total_sources': total_sources,
        'passed': passed_sources,
        'failed': failed_sources,
        'total_records': total_records,
        'total_issues': total_issues,
        'avg_quality_score': float(avg_quality_score)
    },
    'source_results': {}
}

for source_name, result in validation_results.items():
    report_data['source_results'][source_name] = {
        'status': result.get('status'),
        'total_records': result.get('total_records', 0),
        'quality_score': result.get('quality_score', 0),
        'issues': result.get('issues', [])
    }

report_file = f"{REPORT_DIR}/data_quality_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(report_file, 'w') as f:
    json.dump(report_data, f, indent=2)

print(f"\n✅ Quality report saved to: {report_file}")

# Log completion
logger.info("="*80)
logger.info("Data Validation Pipeline - Session Completed")
logger.info(f"Total issues found: {total_issues}")
logger.info(f"Average quality score: {avg_quality_score:.2f}")
logger.info("="*80)

print(f"\n✅ Validation pipeline completed!")
print(f"📝 Full logs available at: {log_file}")